# Workflows vs Autonomous Agents

Not every AI system should decide its own next step. Two dominant patterns compete in production:

| Pattern | Who owns control flow? | Shape |
|---------|------------------------|-------|
| **Workflow** | The engineer (code, graph, rules) | Fixed steps: A → B → C → END |
| **Autonomous agent** | The model (+ guardrails) | Loop: plan → tool → observe → repeat |

**Workflows** are predictable, auditable, and testable node-by-node. **Autonomous agents** adapt when the path is unknown — debugging an incident, exploring a codebase, handling messy user requests.

Most real products are **hybrid**: a workflow shell provides guardrails and compliance checkpoints; autonomous loops handle the hard middle where rigid scripts fail.

This notebook contrasts both patterns with working Python, positions common frameworks conceptually, and explains **when to constrain** model freedom. Everything uses a self-contained **AlertFlow Incident Response** scenario — no prior notebooks required.

In [ ]:
"""
Shared environment: AlertFlow incident response & support.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum, auto
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


RUNBOOKS: list[dict[str, str]] = [
    {
        "id": "rb-401",
        "title": "High error rate on API gateway",
        "text": "Check recent deploys, roll back canary if error rate > 2%, verify upstream health, page on-call if persists > 15 min.",
    },
    {
        "id": "rb-402",
        "title": "Database connection pool exhausted",
        "text": "Inspect slow queries, scale read replicas, restart connection pooler, notify DBA if connections > 90% for 10 min.",
    },
    {
        "id": "rb-403",
        "title": "Webhook delivery failures",
        "text": "Verify endpoint TLS cert, check retry queue depth, test with synthetic payload, escalate if customer endpoint returns 4xx.",
    },
]

INCIDENTS: list[dict[str, Any]] = [
    {
        "id": "INC-1001",
        "alert": "api_gateway_error_rate_high",
        "severity": "high",
        "status": "open",
        "metrics": {"error_rate_pct": 3.2, "region": "us-east-1"},
    },
]

POSTMORTEMS: list[dict[str, Any]] = []

print(f"Runbooks: {len(RUNBOOKS)} | Open incidents: {sum(1 for i in INCIDENTS if i['status'] == 'open')}")

---

## 1. Workflow = Fixed Steps

A **workflow** encodes control flow in **your code**. The model may generate text at specific nodes, but **which node runs next** is not the model's decision.

```
START ──► classify ──► retrieve_runbook ──► draft_update ──► human_review ──► publish ──► END
```

| Property | Implication |
|----------|-------------|
| Edges are **code** | Predictable audits for compliance |
| Branching via **rules** | `if severity == 'critical': page_oncall()` |
| Same path every time (given branch) | Easier SLAs and integration tests |

Classic orchestration: Airflow, AWS Step Functions, or a state graph where transitions are explicit edges — not LLM choices.

In [ ]:
def search_runbooks(query: str, top_k: int = 2) -> list[dict[str, str]]:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    scored = []
    for rb in RUNBOOKS:
        text = f"{rb['title']} {rb['text']}".lower()
        score = sum(1 for term in terms if term in text) if terms else 1
        scored.append((score, rb))
    scored.sort(key=lambda x: -x[0])
    return [{"id": r["id"], "title": r["title"], "text": r["text"]} for _, r in scored[:top_k]]


WORKFLOW_STEPS = ["classify", "retrieve_runbook", "draft_update", "human_review", "publish"]


def run_incident_workflow(incident_id: str) -> dict[str, Any]:
    """Pure workflow — fixed step order, no model routing."""
    incident = next(i for i in INCIDENTS if i["id"] == incident_id)
    trace = []
    ctx: dict[str, Any] = {"incident": incident}

    for step in WORKFLOW_STEPS:
        if step == "classify":
            ctx["category"] = "api_gateway" if "api" in incident["alert"] else "general"
            trace.append({"step": step, "category": ctx["category"]})

        elif step == "retrieve_runbook":
            ctx["runbooks"] = search_runbooks(incident["alert"])
            trace.append({"step": step, "ids": [r["id"] for r in ctx["runbooks"]]})

        elif step == "draft_update":
            rb = ctx["runbooks"][0]
            ctx["draft"] = (
                f"Incident {incident_id} ({incident['severity']}): "
                f"Follow [{rb['id']}] {rb['title']}. "
                f"Current error rate: {incident['metrics'].get('error_rate_pct', 'n/a')}%."
            )
            trace.append({"step": step, "draft_len": len(ctx["draft"])})

        elif step == "human_review":
            ctx["approved"] = incident["severity"] != "critical"  # auto-approve non-critical
            trace.append({"step": step, "approved": ctx["approved"]})

        elif step == "publish":
            if ctx["approved"]:
                POSTMORTEMS.append({"incident_id": incident_id, "summary": ctx["draft"]})
                trace.append({"step": step, "published": True})
            else:
                trace.append({"step": step, "published": False, "reason": "awaiting human"})

    return {"trace": trace, "draft": ctx.get("draft"), "published": ctx.get("approved", False)}


wf_result = run_incident_workflow("INC-1001")
for entry in wf_result["trace"]:
    print(entry)
print(f"\nDraft: {wf_result['draft'][:100]}...")

---

## 2. Autonomous Agent = Model Decides Path

An **autonomous agent** puts the model in the **routing seat** (within guardrails). Each iteration, the model chooses a tool or returns a final answer.

```
        ┌──────────────────────────────────┐
        │  while not done:                  │
        │    model picks tool or final text │
        │    execute tool → observe result  │
        └──────────────────────────────────┘
```

| Property | Implication |
|----------|-------------|
| Next step is **non-deterministic** | Needs `max_steps`, allow-lists, eval harness |
| Great for exploratory tasks | Investigate unknown root cause |
| ReAct-style reasoning | Thought → action → observation cycles |

In [ ]:
def get_incident(incident_id: str) -> dict[str, Any]:
    return next(i for i in INCIDENTS if i["id"] == incident_id)


def check_metrics(alert_name: str) -> dict[str, Any]:
    if "error_rate" in alert_name:
        return {"error_rate_pct": 3.2, "trend": "rising", "last_deploy": "12 min ago"}
    return {"status": "unknown"}


def suggest_rollback(canary_active: bool = True) -> dict[str, str]:
    if canary_active:
        return {"action": "rollback_canary", "command": "deploy rollback --canary"}
    return {"action": "no_canary", "command": "manual investigation required"}


AGENT_TOOLS: dict[str, Callable[..., Any]] = {
    "get_incident": get_incident,
    "search_runbooks": search_runbooks,
    "check_metrics": check_metrics,
    "suggest_rollback": suggest_rollback,
}

ALLOWED_TOOLS = set(AGENT_TOOLS.keys())


class RuleBasedPlanner:
    """Offline stand-in for LLM — decides next tool based on state."""

    def next_action(self, goal: str, observations: list[str], step: int) -> dict[str, Any]:
        if step == 0:
            return {"type": "tool", "name": "get_incident", "args": {"incident_id": "INC-1001"}}
        if step == 1:
            return {"type": "tool", "name": "check_metrics", "args": {"alert_name": "api_gateway_error_rate_high"}}
        if step == 2:
            return {"type": "tool", "name": "search_runbooks", "args": {"query": "api gateway error rate"}}
        if step == 3 and "rising" in " ".join(observations):
            return {"type": "tool", "name": "suggest_rollback", "args": {}}
        return {
            "type": "final",
            "content": "Investigation complete. Recommend rollback if error rate persists.",
        }


@dataclass
class AutonomousInvestigationAgent:
    max_steps: int = 6
    planner: RuleBasedPlanner = field(default_factory=RuleBasedPlanner)
    trace: list[dict[str, Any]] = field(default_factory=list)

    def run(self, goal: str) -> dict[str, Any]:
        self.trace = []
        observations: list[str] = []

        for step in range(self.max_steps):
            action = self.planner.next_action(goal, observations, step)
            self.trace.append({"step": step, "decision": action})

            if action["type"] == "final":
                return {"answer": action["content"], "trace": self.trace, "steps": step + 1}

            name = action["name"]
            if name not in ALLOWED_TOOLS:
                observations.append(f"BLOCKED: {name} not in allow-list")
                continue

            result = AGENT_TOOLS[name](**action["args"])
            obs = pretty(result) if isinstance(result, (dict, list)) else str(result)
            observations.append(obs)
            self.trace[-1]["observation"] = obs[:120]

        return {"answer": "max_steps reached", "trace": self.trace, "steps": self.max_steps}


agent = AutonomousInvestigationAgent(max_steps=5)
agent_result = agent.run("Investigate INC-1001 api gateway errors")
print(f"Answer: {agent_result['answer']}")
print(f"Steps taken: {agent_result['steps']}")
for t in agent_result["trace"]:
    dec = t["decision"]
    print(f"  step {t['step']}: {dec.get('name', 'FINAL')} {dec.get('type')}")

---

## 3. Side-by-Side Comparison

| | **Workflow** | **Autonomous agent** |
|--|--------------|----------------------|
| **Control owner** | Engineer | Model (+ guardrails) |
| **Topology** | DAG / state machine | Loop with tools |
| **Determinism** | Replayable given same inputs | Stochastic across runs |
| **Testing** | Unit test each node | Scenario suites + eval harness |
| **Best for** | Regulated pipelines, cron jobs | Open-ended debugging, messy requests |
| **Compliance** | Easy — graph is the audit log | Harder — must log every model decision |

---

## 4. Hybrid Patterns — Production Default

Most teams ship **hybrids**:

1. **Workflow shell, agent core** — fixed retrieve/classify; autonomous loop only in the middle.
2. **Forced first step** — always load runbook before model may answer.
3. **Supervisor workflow** — manager picks among **fixed** specialist subgraphs.
4. **Human gates on edges** — autonomy only between checkpoints.

```
classify (fixed) ──► retrieve (fixed) ──► agent_tool_loop (autonomous) ──► compliance (fixed) ──► END
```

In [ ]:
def hybrid_incident_pipeline(incident_id: str, max_agent_steps: int = 3) -> dict[str, Any]:
    """
    Hybrid: fixed classify + retrieve, then short autonomous tool loop, then fixed publish gate.
    """
    trace = []
    incident = get_incident(incident_id)

    # Fixed workflow phases
    trace.append({"phase": "workflow", "step": "classify", "severity": incident["severity"]})
    runbooks = search_runbooks(incident["alert"])
    trace.append({"phase": "workflow", "step": "retrieve", "ids": [r["id"] for r in runbooks]})

    # Autonomous middle — agent explores metrics and rollback options
    agent = AutonomousInvestigationAgent(max_steps=max_agent_steps)
    investigation = agent.run(f"Investigate {incident_id}")
    for t in investigation["trace"]:
        trace.append({"phase": "agent", "step": t["step"], "decision": t["decision"].get("name", "final")})

    # Fixed compliance gate
    approved = incident["severity"] != "critical"
    trace.append({"phase": "workflow", "step": "human_review", "approved": approved})

    summary = f"[{runbooks[0]['id']}] {investigation['answer']}"
    if approved:
        POSTMORTEMS.append({"incident_id": incident_id, "summary": summary})
        trace.append({"phase": "workflow", "step": "publish", "done": True})

    return {"trace": trace, "summary": summary, "approved": approved}


hybrid = hybrid_incident_pipeline("INC-1001")
print(pretty(hybrid["trace"]))
print(f"\nSummary: {hybrid['summary']}")

---

## 5. State Machine vs Agent Loop

| State machine (workflow) | Agent loop |
|--------------------------|------------|
| States are **enumerated** (`RETRIEVE`, `DRAFT`, …) | State is message history + tool outputs |
| Transitions are **labeled events** | Transitions are **model decisions** |
| Deterministic replay | Stochastic replay |
| Engineer adds edges | Engineer sets guardrails only |

Frameworks like LangGraph can represent **both**: fixed edges for workflows, cyclic agent↔tools edges for autonomy.

In [ ]:
class IncidentWorkflowState(Enum):
    CLASSIFY = auto()
    RETRIEVE = auto()
    DRAFT = auto()
    REVIEW = auto()
    PUBLISH = auto()
    DONE = auto()


TRANSITIONS: dict[IncidentWorkflowState, IncidentWorkflowState] = {
    IncidentWorkflowState.CLASSIFY: IncidentWorkflowState.RETRIEVE,
    IncidentWorkflowState.RETRIEVE: IncidentWorkflowState.DRAFT,
    IncidentWorkflowState.DRAFT: IncidentWorkflowState.REVIEW,
    IncidentWorkflowState.REVIEW: IncidentWorkflowState.PUBLISH,
    IncidentWorkflowState.PUBLISH: IncidentWorkflowState.DONE,
}


def run_state_machine(start: IncidentWorkflowState = IncidentWorkflowState.CLASSIFY) -> list[str]:
    path = [start.name]
    state = start
    while state != IncidentWorkflowState.DONE:
        state = TRANSITIONS.get(state, IncidentWorkflowState.DONE)
        path.append(state.name)
    return path


print("Workflow state path:", " → ".join(run_state_machine()))

---

## 6. Framework Topologies (Conceptual)

Different frameworks bias toward workflow or autonomy — you can usually configure either:

| Topology | Control flow | Framework mapping |
|----------|--------------|-------------------|
| **Pure workflow** | START → retrieve → write → END | LangGraph linear graph; CrewAI sequential |
| **Agentic loop** | START → agent ↔ tools → … → END | LangGraph cyclic edges; ReAct agents |
| **Supervisor** | START → supervisor → worker → supervisor → END | LangGraph conditional routing; AutoGen selector chat |

```python
# Conceptual LangGraph — pure workflow
# builder.add_edge(START, "retrieve")
# builder.add_edge("retrieve", "write")
# builder.add_edge("write", END)

# Add autonomy: replace "write" with agent ↔ tools cycle
```

Pick the abstraction that matches how your **team** reasons about the product — graphs for auditable flows, conversation for exploratory work.

In [ ]:
TOPOLOGIES = {
    "pure_workflow": ["START", "classify", "retrieve", "draft", "publish", "END"],
    "agentic_loop": ["START", "agent", "tools", "agent", "...", "END"],
    "supervisor": ["START", "supervisor", "worker_a|worker_b", "supervisor", "END"],
    "hybrid": ["START", "classify", "retrieve", "agent_loop", "review", "END"],
}

for name, nodes in TOPOLOGIES.items():
    print(f"{name:16} {' → '.join(nodes)}")

---

## 7. The Autonomy Slider — Levels 0 to 4

Autonomy is a **slider**, not a boolean:

| Level | Description | AlertFlow example |
|-------|-------------|-------------------|
| **0** | Pure workflow, no model routing | Nightly runbook sync cron |
| **1** | Model writes text only; tools triggered by code | Template-filled status page update |
| **2** | Model picks from allowlisted tools | Debug agent: metrics + runbook search only |
| **3** | Model picks tools + proposes sub-goals | Multi-step investigation with rollback suggestion |
| **4** | Multi-agent delegation | Supervisor + SRE + comms agents (highest risk) |

In [ ]:
AUTONOMY_LEVELS: dict[int, dict[str, Any]] = {
    0: {"model_routes": False, "tools": "code-only", "example": "cron runbook sync"},
    1: {"model_routes": False, "tools": "code-triggered", "example": "draft incident update text"},
    2: {"model_routes": True, "tools": "allowlisted", "example": "metrics + runbook search"},
    3: {"model_routes": True, "tools": "allowlisted + sub-goals", "example": "investigate + suggest rollback"},
    4: {"model_routes": True, "tools": "multi-agent", "example": "supervisor + specialists"},
}


def recommend_autonomy_level(task: str) -> int:
    t = task.lower()
    if "nightly" in t or "cron" in t or "sync" in t:
        return 0
    if "draft" in t and "template" in t:
        return 1
    if "debug" in t or "investigate" in t:
        return 3
    if "multi-team" in t or "war room" in t:
        return 4
    return 2


for task in ["nightly runbook sync", "investigate api errors", "war room coordination"]:
    level = recommend_autonomy_level(task)
    print(f"Level {level}: {task} — {AUTONOMY_LEVELS[level]['example']}")

---

## 8. When to Constrain Autonomy

Constrain when **mistakes are expensive** or **paths must be auditable**:

| Risk | Constraint |
|------|------------|
| Production writes (rollback, delete) | Human approval node before execute |
| PII / customer data exposure | Allowlist tools; block free browsing |
| Runaway loops | `max_iterations`, token budget, wall-clock timeout |
| Regulatory trace | Fixed graph + logged node inputs/outputs |
| Irreversible actions | Workflow gate — agent may *propose*, not *execute* |

In [ ]:
GUARDRAILS = {
    "max_tool_calls": 5,
    "allowed_tools": ["get_incident", "search_runbooks", "check_metrics", "suggest_rollback"],
    "blocked_tools": ["drop_database", "shell", "delete_incident"],
    "require_human_for": ["execute_rollback", "publish_postmortem", "page_executive"],
}


def tool_policy(name: str) -> str:
    if name in GUARDRAILS["blocked_tools"]:
        return "DENY"
    if name in GUARDRAILS["require_human_for"]:
        return "HUMAN_APPROVAL"
    if name in GUARDRAILS["allowed_tools"]:
        return "ALLOW"
    return "DENY"


for tool in ["search_runbooks", "execute_rollback", "drop_database", "check_metrics"]:
    print(f"{tool:22} → {tool_policy(tool)}")

---

## 9. Hybrid Support Bot — Intent Routing

A common production pattern: **workflow routes** to different paths; only some paths contain autonomous loops.

```
START → classify_intent (workflow)
     → faq_path      → retrieve → answer → END
     → debug_path    → retrieve → agent_tool_loop → summarize → END
     → escalate_path → human → END
```

In [ ]:
def route_intent(text: str) -> str:
    t = text.lower()
    if any(w in t for w in ("error", "401", "500", "down", "investigate")):
        return "debug_path"
    if any(w in t for w in ("runbook", "how do i", "webhook", "rollback steps")):
        return "faq_path"
    return "escalate_path"


def run_support_router(user_message: str) -> dict[str, Any]:
    path = route_intent(user_message)
    trace = [{"router": path}]

    if path == "faq_path":
        hits = search_runbooks(user_message)
        answer = f"[{hits[0]['id']}] {hits[0]['text']}" if hits else "No runbook found."
        trace.append({"workflow": "retrieve + answer"})
        return {"path": path, "answer": answer, "trace": trace}

    if path == "debug_path":
        agent = AutonomousInvestigationAgent(max_steps=3)
        result = agent.run(user_message)
        trace.append({"agent_steps": result["steps"]})
        return {"path": path, "answer": result["answer"], "trace": trace}

    return {"path": path, "answer": "Connecting you with on-call engineer.", "trace": trace}


for msg in [
    "What are the rollback steps for api gateway errors?",
    "Error rate spiking on gateway — need investigation",
    "What's your enterprise pricing?",
]:
    out = run_support_router(msg)
    print(f"[{out['path']}] {msg[:50]}...")
    print(f"  → {out['answer'][:70]}...\n")

---

## 10. Termination Conditions for Autonomous Loops

Every autonomous loop needs **explicit stop rules** — never rely on the model to stop on its own:

- `max_iterations` reached
- Model returns final text (no tool calls)
- Stop keyword (`FINAL ANSWER`, `APPROVED`)
- External cancel or human interrupt
- Cost or token budget exceeded

In [ ]:
def should_stop(
    messages: list[dict[str, Any]],
    step: int,
    max_steps: int = 5,
    token_budget: int = 8000,
    tokens_used: int = 0,
) -> tuple[bool, str]:
    if step >= max_steps:
        return True, "max_steps"
    if tokens_used >= token_budget:
        return True, "token_budget"
    if not messages:
        return False, "continue"
    last = messages[-1]
    if last.get("role") == "assistant" and not last.get("tool_calls"):
        return True, "final_message"
    content = last.get("content") or ""
    if "FINAL ANSWER" in content or "APPROVED" in content:
        return True, "stop_keyword"
    return False, "continue"


test_cases = [
    ([{"role": "assistant", "content": "FINAL ANSWER: rollback canary"}], 2),
    ([{"role": "assistant", "tool_calls": [{"name": "check_metrics"}]}], 1),
    ([], 6),
]
for msgs, step in test_cases:
    stop, reason = should_stop(msgs, step, max_steps=5)
    print(f"step={step} → stop={stop} ({reason})")

---

## 11. Checkpointing — Workflow Superpower

**Workflows** excel at **resumability**: save state after each node, pause for human input, continue after approval. This is harder with unconstrained agent loops where state is an unbounded message history.

| Capability | Workflow | Autonomous agent |
|------------|----------|------------------|
| Pause at known node | Natural (`human_review`) | Must define checkpoint semantics |
| Resume after crash | Replay from last node | Truncate or summarize history |
| Audit trail | Node inputs/outputs | Full message trace |

Hybrid design: checkpoint at **workflow boundaries**; treat agent loop as a bounded subgraph with its own `max_steps`.

In [ ]:
@dataclass
class WorkflowCheckpoint:
    """Simulates persisted workflow state between nodes."""
    run_id: str
    current_node: str
    payload: dict[str, Any] = field(default_factory=dict)
    saved_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


def simulate_pause_and_resume(incident_id: str) -> list[str]:
    log = []
    cp = WorkflowCheckpoint(run_id=str(uuid.uuid4())[:8], current_node="retrieve", payload={"incident_id": incident_id})
    log.append(f"checkpoint saved at {cp.current_node}")

    # Simulate human delay...
    cp.current_node = "human_review"
    cp.payload["draft"] = "Rollback canary recommended."
    log.append(f"resumed at {cp.current_node} with draft ready")

    cp.current_node = "publish"
    log.append(f"completed {cp.current_node}")
    return log


for line in simulate_pause_and_resume("INC-1001"):
    print(line)

---

## 12. Same Task — Workflow vs Agent vs Hybrid

Run the same incident through all three patterns and compare traces.

In [ ]:
def compare_patterns(incident_id: str) -> dict[str, Any]:
    workflow = run_incident_workflow(incident_id)
    agent = AutonomousInvestigationAgent(max_steps=4).run(f"Investigate {incident_id}")
    hybrid = hybrid_incident_pipeline(incident_id, max_agent_steps=2)

    return {
        "workflow": {"steps": len(workflow["trace"]), "deterministic": True},
        "agent": {"steps": agent["steps"], "deterministic": False},
        "hybrid": {"steps": len(hybrid["trace"]), "deterministic": "partial"},
    }


comparison = compare_patterns("INC-1001")
print(f"{'Pattern':<12} {'Steps':>6}  Deterministic")
print("-" * 40)
for name, info in comparison.items():
    print(f"{name:<12} {info['steps']:>6}  {info['deterministic']}")

---

## 13. Choosing for AlertFlow Tasks

| Task | Recommendation | Autonomy level |
|------|----------------|----------------|
| Nightly runbook index sync | Pure workflow | 0 |
| Template status page update | Workflow + LLM draft | 1 |
| Interactive error investigation | Hybrid | 2–3 |
| War-room multi-team response | Multi-agent under supervisor | 4 |
| Execute production rollback | Workflow + human approval | 0 at execute step |

In [ ]:
def classify_task(task: str) -> tuple[str, int]:
    t = task.lower()
    if "nightly" in t or "cron" in t or "sync" in t:
        return "pure_workflow", 0
    if "rollback" in t and "execute" in t:
        return "workflow_with_hitl", 0
    if "debug" in t or "investigate" in t:
        return "hybrid", 3
    if "war room" in t:
        return "multi_agent_supervisor", 4
    return "hybrid", 2


TASKS = [
    "nightly runbook sync",
    "debug 502 errors on webhook delivery",
    "execute rollback on production canary",
    "war room for regional outage",
]
print(f"{'Task':<42} {'Pattern':<25} Level")
print("-" * 72)
for task in TASKS:
    pattern, level = classify_task(task)
    print(f"{task:<42} {pattern:<25} {level}")

---

## 14. Framework Perspectives (Without Lock-In)

| Framework | Default bias | How to constrain autonomy |
|-----------|--------------|---------------------------|
| **LangGraph** | Workflow graph | Fixed edges; cap cycles; `interrupt` before dangerous nodes |
| **AutoGen** | Conversation autonomy | Termination conditions; limited tool lists |
| **CrewAI** | Role workflow (`Process.sequential`) | Manager agent + fixed task order |
| **Plain Python** | Whatever you code | This notebook's approach — full control |

The pattern matters more than the framework: **define control flow explicitly**, then decide which nodes (if any) delegate routing to the model.

---

## 15. Optional — Live LLM Perspective

Set `USE_LIVE_LLM = True` with a valid API key for a one-sentence comparison.

In [ ]:
USE_LIVE_LLM = False

OFFLINE_ANSWER = (
    "Workflows fix control flow in code for predictability; autonomous agents let the model "
    "choose tools within guardrails — production systems combine both in hybrid pipelines."
)

if USE_LIVE_LLM:
    try:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": "Workflow vs autonomous agent in one sentence."}],
            max_tokens=60,
        )
        print(resp.choices[0].message.content)
    except Exception as exc:
        print("LLM call failed:", exc)
else:
    print(OFFLINE_ANSWER)

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Full autonomy for regulated steps | Audit failures, irreversible errors | Workflow gates + HITL |
| Pure workflow for messy debugging | Brittle rules, constant code changes | Hybrid with agent middle |
| No `max_steps` on agent loops | Runaway cost | Hard cap + budget |
| Agent proposes AND executes rollback | Production incidents | Separate propose vs execute nodes |
| Ignoring checkpointing | Cannot resume after human delay | Persist state at workflow nodes |

---

## 17. Check Your Understanding

1. Who owns control flow in a workflow vs an autonomous agent?
2. Draw the hybrid pipeline from section 4 in your own words.
3. What autonomy level fits a nightly cron job vs an interactive investigation?
4. Why does `execute_rollback` map to `HUMAN_APPROVAL` in the tool policy?
5. What are three valid termination conditions for an agent loop?
6. Why are workflows easier to checkpoint than unconstrained agent loops?
7. When would you choose `debug_path` vs `faq_path` in the support router?

---

## 18. Summary

**Workflows** fix control flow in code — predictable, auditable, resumable. **Autonomous agents** delegate routing to the model within guardrails — flexible for exploratory work. **Production systems are usually hybrid.**

**Key takeaways:**

- Match pattern to **task shape**: cron jobs → workflow; debugging → hybrid; war rooms → supervised multi-agent.
- **Autonomy is a slider** (levels 0–4), not a boolean — constrain as risk increases.
- **Guardrails**: allow-listed tools, `max_steps`, human gates for irreversible actions.
- **Intent routing** lets workflows handle simple paths and agents handle hard paths in one product.
- **Checkpointing** at workflow nodes enables pause/resume; bound agent loops inside those nodes.
- Frameworks implement the same ideas — graphs for workflows, cycles for agents — pick what your team can operate.

For any new feature, ask: *Which steps must be fixed for compliance, and where is model-driven routing worth the risk?*